In [1]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# 准备测试数据 ，假设我们提供的文档数据如下：
documents = [
    Document(
        page_content="狗是伟大的伴侣，以其忠诚和友好而闻名。",
        metadata={"source": "哺乳动物宠物文档"},
    ),
    Document(
        page_content="猫是独立的宠物，通常喜欢自己的空间。",
        metadata={"source": "哺乳动物宠物文档"},
    ),
    Document(
        page_content="金鱼是初学者的流行宠物，需要相对简单的护理。",
        metadata={"source": "鱼类宠物文档"},
    ),
    Document(
        page_content="鹦鹉是聪明的鸟类，能够模仿人类的语言。",
        metadata={"source": "鸟类宠物文档"},
    ),
    Document(
        page_content="兔子是社交动物，需要足够的空间跳跃。",
        metadata={"source": "哺乳动物宠物文档"},
    ),
]


In [2]:
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 实例化一个向量数据库
vector_store = Chroma.from_documents(documents, embedding=OpenAIEmbeddings(
    model='Qwen/Qwen3-Embedding-8B',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
))

In [3]:
vector_store.similarity_search_with_score('小兔子')

[(Document(id='42e5f1a0-fb04-40e5-9bfa-b58b4bab9c3a', metadata={'source': '哺乳动物宠物文档'}, page_content='兔子是社交动物，需要足够的空间跳跃。'),
  1.124269723892212),
 (Document(id='bcec37da-5b01-4358-983f-dd80a7bdab84', metadata={'source': '鸟类宠物文档'}, page_content='鹦鹉是聪明的鸟类，能够模仿人类的语言。'),
  1.1281665563583374),
 (Document(id='c47d9525-a49d-4d04-829c-9557b3570af5', metadata={'source': '哺乳动物宠物文档'}, page_content='狗是伟大的伴侣，以其忠诚和友好而闻名。'),
  1.1477338075637817),
 (Document(id='1f0cb22f-6597-44a3-88a7-adc3a3a2e1aa', metadata={'source': '鱼类宠物文档'}, page_content='金鱼是初学者的流行宠物，需要相对简单的护理。'),
  1.1509981155395508)]

In [4]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 根据 vector_store 创建一个检索器  bind(k=1) 表示只返回最相似的1个结果
retriever = RunnableLambda(vector_store.similarity_search).bind(k=1)

In [5]:
retriever.batch(["小兔子", "人类的"])

[[Document(id='42e5f1a0-fb04-40e5-9bfa-b58b4bab9c3a', metadata={'source': '哺乳动物宠物文档'}, page_content='兔子是社交动物，需要足够的空间跳跃。')],
 [Document(id='bcec37da-5b01-4358-983f-dd80a7bdab84', metadata={'source': '鸟类宠物文档'}, page_content='鹦鹉是聪明的鸟类，能够模仿人类的语言。')]]

In [6]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='Qwen/Qwen3-32B',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

message = """
使用提供的上下文仅回答这个问题：
{question}
上下文:
{context}
"""
prompt_temp = ChatPromptTemplate.from_messages([('human', message)])

# RunnablePassthrough 表示将输入传递给下一个步骤
chain = {'question': RunnablePassthrough(), 'context': retriever} | prompt_temp | model

In [7]:
def stream_chat(resp_stream):
    result = ""
    for chunk in resp_stream:
        # 一个chunk就是一个token
        print(chunk.content)
        result += chunk.content

In [8]:
stream_chat(chain.stream('请介绍一下猫'))
























































































































































































提供的
上下
文中
并未
包含
关于
猫
的
任何
信息
，
该
文档
仅
描述
了
兔子
的
社交
性
及
活动
需求
（
“
兔子
是
社交
动物
，
需要
足够的
空间
跳跃
”
）。
我
无法
基于
此
内容
回答
关于
猫
的问题
。
如
需
了解
猫
的
介绍
，
建议
提供
包含
猫
的相关
资料
的
上下
文
。


